In [1]:
%cd ../../../

/Users/hoangle/Projects/Food-Waste-Optimization


In [2]:
# from itertools import product

import numpy as np
import polars as pl
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor
# from lightgbm import LGBMRegressor
# from sklearn.metrics import root_mean_squared_error, r2_score
import lightning as L
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torch
from torch.nn import Module
from torch import Tensor
from torchmetrics.regression import MeanSquaredError, R2Score
from lightning.pytorch.callbacks import RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.preprocessing import MinMaxScaler
from polars import DataFrame

# Load data and resources

In [3]:
path = "data/processed/phase_4/dim_pieces_per_dish.xlsx"

pos = pl.read_excel(path)
pos.head()

date,restaurant,meal_type,pcs,meal_id
date,str,str,i64,i64
2023-01-02,"""che""","""fish""",78,9500047
2023-01-02,"""che""","""vegan""",84,6128
2023-01-02,"""che""","""meat""",165,9500139
2023-01-03,"""che""","""vegetarian""",29,1270
2023-01-03,"""che""","""fish""",105,6156


In [4]:
path = "notebooks/phase_4/2_foreacast_pieces_per_dish/data/Nov22_meals_cosine_sim.parquet"
cosine_sim = pl.read_parquet(path)

cosine_sim.head()

meal_id_x,meal_id_y,cosine_sim,__index_level_0__
i64,i64,f64,i64
34,37,0.040771,1
34,710,0.072106,2
34,713,0.031979,3
34,724,0.063252,4
34,725,0.026991,5


In [5]:
path = "data/processed/phase_4/dim_meals.xlsx"

dim_meals = pl.read_excel(path)
dim_meals.head()

meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
i64,str,str,bool,bool,str,str,f64
9017,"""vegan""","""24-25""",true,true,"""che-exa""","""vegan-miscellaneous""",140.165854
7201,"""vegan""","""23-24""",false,false,null,null,121.918212
9032,"""vegan""","""23-24""",false,false,null,null,121.918212
9102,"""vegan""","""23-24""",false,false,null,null,121.918212
7010,"""vegetarian""","""24-25""",false,false,"""che-exa""",null,71.0


## Process dim tables

In [6]:
# Filter out too low POS values
THETA = 5
pos = pos.filter(pl.col('pcs') >= THETA)

# Keep THETA most consumed meals per date-restaurant
THETA = 5
pos = (
    pos
    .with_columns(pl.col('pcs').rank(descending=True, method='dense').over("date", 'restaurant').alias('rank'))
    .filter(pl.col("rank") <= THETA)
    .drop('rank')
)


pos.head()

date,restaurant,meal_type,pcs,meal_id
date,str,str,i64,i64
2023-01-02,"""che""","""fish""",78,9500047
2023-01-02,"""che""","""vegan""",84,6128
2023-01-02,"""che""","""meat""",165,9500139
2023-01-03,"""che""","""vegetarian""",29,1270
2023-01-03,"""che""","""fish""",105,6156


In [7]:
dim_meals = (
    dim_meals
    .filter(pl.col('restaurant').is_not_null())
    .select(
        'meal_id',
        pl.col('meal_type_1').alias('meal_type'),
        # pl.col('restaurant').str.split('-')
    )
)

dim_meals.head()

meal_id,meal_type
i64,str
9017,"""vegan"""
7010,"""vegetarian"""
6557,"""vegan"""
1751,"""chicken"""
9039,"""vegan"""


# Craft feature

Idea: Refer every meals to `K` highest sale meals per restaurant - meal-type

## Get `K` highest sale meals per restaurant - meal_type

In [8]:
K = 5
CUTOFF_DATE = "2024-10-01"

pos_train = pos.filter(pl.col('date') < pl.lit(CUTOFF_DATE).str.to_datetime())

In [9]:
meals_topK = (
    pos_train
    .group_by('restaurant', 'meal_type', 'meal_id')
    .agg(pl.col('pcs').sum())
    .with_columns(
        pl.col('pcs').rank(method='dense', descending=True).over('restaurant', 'meal_type').alias('rank')
    )
    .filter(pl.col('rank') <= K)
    .drop('rank', 'pcs')
)

meals_topK.head()

restaurant,meal_type,meal_id
str,str,i64
"""che""","""vegan""",6673
"""exa""","""vegan""",9500062
"""phy""","""vegan""",6353
"""che""","""vegetarian""",6658
"""phy""","""meat""",1515


In [10]:
meals_type = dim_meals.select('meal_id', pl.col('meal_type'))
meal_ids_valid = meals_topK.get_column('meal_id').unique()

In [14]:
cosine_sim.head()

meal_id_x,meal_id_y,cosine_sim,__index_level_0__
i64,i64,f64,i64
34,37,0.040771,1
34,710,0.072106,2
34,713,0.031979,3
34,724,0.063252,4
34,725,0.026991,5


In [ ]:
meals_similar = (
    cosine_sim

    # Only consider meals appearing in POS data
    .filter(pl.col('meal_id_y').is_in(meal_ids_valid))

    .join(meals_type, left_on='meal_id_x', right_on='meal_id', how='left')
    .rename({'meal_type': 'meal_type_x'})
    .join(meals_type, left_on='meal_id_y', right_on='meal_id', how='left')
    .filter(pl.col('meal_type_x') == pl.col('meal_type'))

    .with_columns(
        pl.col('cosine_sim').rank('dense', descending=True).over('meal_id_x').alias('rank')
    )
    .filter(pl.col("rank") <= K)
    .select('meal_id_x', 'meal_id_y', 'cosine_sim')
)


# Add entries for each meal with itself
ids = meals_similar.select(pl.col('meal_id_x').unique()).get_column('meal_id_x')
tmp = pl.DataFrame({
    'meal_id_x': ids,
    'meal_id_y': ids,
    'cosine_sim': 1.0
})
meals_similar_all = pl.concat([meals_similar, tmp])

